# Customer Segmentation for Retail Loyalty Data

**Portfolio notebook**  
This notebook builds a customer segmentation solution for a convenience-retail loyalty-card dataset. It converts basket-level and category-level transaction data into customer-level behavioural features, applies cleaning and feature engineering, reduces dimensionality using PCA, and fits a K-Means clustering model to create actionable customer segments.

> **Data notice:** raw coursework data is not included in the public portfolio repository. The notebook is provided to demonstrate the analytical workflow and can be run if the four source CSV files are placed in the expected local folder.

## 1. Business objective

The objective is to help a retailer understand its loyalty-card customer base and support targeted marketing decisions. The segmentation is designed to identify groups that differ by:

- shopping frequency and recency,
- total and average basket spend,
- basket size and category breadth,
- category-level purchasing patterns.

The output is a set of segment assignments and customer profiles that can be used for campaign targeting, retention planning and basket-building initiatives.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 2. File paths and configuration

Expected data files:

- `customers_sample.csv`
- `baskets_sample.csv`
- `category_spends_sample.csv`
- `lineitems_sample.csv`



In [ ]:

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../outputs")
FIGURE_DIR = Path("../figures")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CORR_THRESHOLD = 0.95
PCA_VARIANCE_TARGET = 0.80
K_RANGE = range(4, 11)
K_FINAL = 5

file_map = {
    "customers": DATA_DIR / "customers_sample.csv",
    "baskets": DATA_DIR / "baskets_sample.csv",
    "category_spends": DATA_DIR / "category_spends_sample.csv",
    "lineitems": DATA_DIR / "lineitems_sample.csv",
}

missing = [str(path) for path in file_map.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Raw data files are not included in this public portfolio notebook. "
        "Place the required CSV files in ../data/raw/ to run the notebook. Missing files: "
        + ", ".join(missing)
    )

## 3. Load data

In [ ]:
df_customer = pd.read_csv(file_map["customers"])
df_basket = pd.read_csv(file_map["baskets"])
df_category = pd.read_csv(file_map["category_spends"])
df_items = pd.read_csv(file_map["lineitems"])

print("Loaded shapes:")
print("customers:", df_customer.shape)
print("baskets:", df_basket.shape)
print("category spends:", df_category.shape)
print("lineitems:", df_items.shape)

## 4. Cleaning utilities

Spend columns may appear as currency strings. These are converted into numeric values. Negative spend and quantity values are treated as transaction corrections/returns and set to zero for segmentation purposes, so clusters reflect purchasing behaviour rather than accounting adjustments.

In [ ]:
def clean_currency(series: pd.Series) -> pd.Series:
    """Convert a currency-like series to numeric float values."""
    if series.dtype == "object":
        cleaned = (
            series.astype(str)
            .str.replace("£", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        return pd.to_numeric(cleaned, errors="coerce")
    return pd.to_numeric(series, errors="coerce")


def floor_negative_values(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Replace negative numeric values with zero in selected columns."""
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
            df.loc[df[col] < 0, col] = 0
    return df

In [ ]:
# Convert known monetary fields
for col in ["total_spend", "average_spend"]:
    if col in df_customer.columns:
        df_customer[col] = clean_currency(df_customer[col])

if "basket_spend" in df_basket.columns:
    df_basket["basket_spend"] = clean_currency(df_basket["basket_spend"])

if "spend" in df_items.columns:
    df_items["spend"] = clean_currency(df_items["spend"])

category_cols = [c for c in df_category.columns if c != "customer_number"]
for col in category_cols:
    df_category[col] = clean_currency(df_category[col])

# Floor negative values where they would distort clustering distances
basket_numeric = ["basket_quantity", "basket_spend", "basket_categories"]
item_numeric = ["quantity", "spend"]
df_basket = floor_negative_values(df_basket, basket_numeric)
df_items = floor_negative_values(df_items, item_numeric)
df_category = floor_negative_values(df_category, category_cols)

# Parse purchase timestamp
if "purchase_time" in df_basket.columns:
    df_basket["purchase_time"] = pd.to_datetime(df_basket["purchase_time"], errors="coerce")
if "purchase_time" in df_items.columns:
    df_items["purchase_time"] = pd.to_datetime(df_items["purchase_time"], errors="coerce")

## 5. Bakery spend correction

The original category-spend table recorded bakery spend as zero for all customers, while line-item data contained bakery purchases. The bakery feature is recalculated from the line-item table and used to replace the incorrect category-spend values.

In [ ]:
# Keep the original bakery value for audit only
if "bakery" in df_category.columns:
    df_category["bakery_original"] = df_category["bakery"]

bakery_from_items = (
    df_items[df_items["category"].astype(str).str.upper().eq("BAKERY")]
    .groupby("customer_number")["spend"]
    .sum()
)

df_category["bakery"] = df_category["customer_number"].map(bakery_from_items).fillna(0.0)

bakery_diff = (df_category["bakery"] - df_category.get("bakery_original", 0)).abs()
bakery_audit = pd.DataFrame({
    "metric": [
        "customers_total",
        "customers_changed",
        "pct_changed",
        "bakery_original_mean",
        "bakery_corrected_mean",
    ],
    "value": [
        len(df_category),
        int((bakery_diff > 1e-6).sum()),
        round((bakery_diff > 1e-6).mean() * 100, 1),
        round(df_category.get("bakery_original", pd.Series([0])).mean(), 2),
        round(df_category["bakery"].mean(), 2),
    ],
})

bakery_audit

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df_category["bakery"], bins=40)
plt.title("Bakery spend recalculated from line items")
plt.xlabel("Bakery spend")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "bakery_spend_distribution.png", dpi=150)
plt.show()

## 6. Customer-level feature engineering

Basket-level records are aggregated into one row per customer. The features capture engagement, spend intensity, basket structure and product/category preference.

In [ ]:
basket_agg = (
    df_basket.groupby("customer_number")
    .agg(
        n_baskets=("purchase_time", "count"),
        total_basket_qty=("basket_quantity", "sum"),
        avg_basket_qty=("basket_quantity", "mean"),
        std_basket_qty=("basket_quantity", "std"),
        total_basket_spend=("basket_spend", "sum"),
        avg_basket_spend=("basket_spend", "mean"),
        std_basket_spend=("basket_spend", "std"),
        max_basket_spend=("basket_spend", "max"),
        avg_basket_categories=("basket_categories", "mean"),
        std_basket_categories=("basket_categories", "std"),
        active_days=("purchase_time", lambda s: s.dt.date.nunique()),
        first_purchase=("purchase_time", "min"),
        last_purchase=("purchase_time", "max"),
    )
    .reset_index()
)

max_date = df_basket["purchase_time"].max()
basket_agg["recency_days"] = (max_date - basket_agg["last_purchase"]).dt.days
basket_agg["tenure_days"] = (basket_agg["last_purchase"] - basket_agg["first_purchase"]).dt.days.clip(lower=0)
basket_agg = basket_agg.drop(columns=["first_purchase", "last_purchase"])

model_df = (
    df_customer
    .merge(basket_agg, on="customer_number", how="left")
    .merge(df_category, on="customer_number", how="left")
)

# Remove audit-only field from modelling table
model_df = model_df.drop(columns=["bakery_original"], errors="ignore")

# Replace standard-deviation NaNs from customers with only one basket
std_cols = [c for c in model_df.columns if c.startswith("std_")]
model_df[std_cols] = model_df[std_cols].fillna(0.0)

# Fill remaining numeric missing values conservatively
numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
model_df[numeric_cols] = model_df[numeric_cols].fillna(0.0)

print("Customer-level modelling table:", model_df.shape)
model_df.head()

In [ ]:
# Correlation heatmap for the most connected features
corr = X.corr(numeric_only=True)
top_features = corr.abs().sum().sort_values(ascending=False).head(20).index
corr_top = corr.loc[top_features, top_features]

plt.figure(figsize=(10, 8))
plt.imshow(corr_top, aspect="auto")
plt.xticks(range(len(top_features)), top_features, rotation=90)
plt.yticks(range(len(top_features)), top_features)
plt.colorbar(label="Correlation")
plt.title("Correlation heatmap: top 20 features")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "correlation_heatmap_top20.png", dpi=150)
plt.show()

## 7. Feature selection and redundancy control

Obvious duplicate summary fields are removed because they duplicate information already captured through basket aggregations. Highly correlated fields are then screened to avoid double-counting the same behavioural signal in the distance-based clustering model.

In [ ]:
redundant_cols = [
    "baskets",            # duplicates n_baskets
    "total_quantity",     # duplicates total_basket_qty
    "average_quantity",   # duplicates avg_basket_qty
    "total_spend",        # duplicates total_basket_spend
    "average_spend",      # duplicates avg_basket_spend
]

feature_df = model_df.drop(columns=[c for c in redundant_cols if c in model_df.columns])

X = feature_df.drop(columns=["customer_number"])

corr = X.corr(numeric_only=True)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

candidate_drop = [col for col in upper.columns if any(upper[col].abs() > CORR_THRESHOLD)]
print("Candidate highly correlated drops:", candidate_drop)

# Keep behaviourally interpretable fields for the report and drop less useful overlap fields
manual_drop = ["tenure_days", "active_days"]
X = X.drop(columns=[c for c in manual_drop if c in X.columns])

print("Final modelling feature count:", X.shape[1])

## 8. Scaling and PCA

K-Means uses distances, so all features are standardised before clustering. PCA is used to reduce dimensionality and residual collinearity while retaining most of the behavioural signal.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA(n_components=None, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

def components_for_variance(target: float) -> int:
    return int(np.where(cumulative >= target)[0][0] + 1)

n_components = components_for_variance(PCA_VARIANCE_TARGET)
print(f"Components needed for {PCA_VARIANCE_TARGET:.0%} variance:", n_components)
print("Components needed for 90% variance:", components_for_variance(0.90))

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
plt.axhline(PCA_VARIANCE_TARGET, linestyle="--", linewidth=1)
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pca_explained_variance.png", dpi=150)
plt.show()

## 9. Select number of clusters

Candidate `k` values are compared using inertia and silhouette score. The final solution uses five clusters because it gave the best balance between diagnostic performance and interpretable segment profiles.

In [ ]:
X_cluster = X_pca_full[:, :n_components]

inertias = []
silhouette_scores = []

for k in K_RANGE:
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_cluster)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster, labels))

k_eval = pd.DataFrame({
    "k": list(K_RANGE),
    "inertia": inertias,
    "silhouette_score": silhouette_scores,
})

k_eval

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(k_eval["k"], k_eval["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "elbow_curve.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(k_eval["k"], k_eval["silhouette_score"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette score by k")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "silhouette_scores.png", dpi=150)
plt.show()

## 10. Fit final K-Means model

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, n_init=20, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(X_cluster)

final_silhouette = silhouette_score(X_cluster, cluster_labels)
print(f"Final k={K_FINAL} silhouette score: {final_silhouette:.3f}")

silhouette_values = silhouette_samples(X_cluster, cluster_labels)
y_lower = 10

plt.figure(figsize=(7, 5))
for i in range(K_FINAL):
    values = silhouette_values[cluster_labels == i]
    values.sort()
    y_upper = y_lower + len(values)
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, values, alpha=0.8)
    plt.text(-0.05, y_lower + 0.5 * len(values), str(i))
    y_lower = y_upper + 10

plt.axvline(final_silhouette, linestyle="--", linewidth=1)
plt.xlabel("Silhouette coefficient")
plt.ylabel("Cluster")
plt.title("Silhouette plot for final k=5 solution")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_silhouette_plot.png", dpi=150)
plt.show()

## 11. Segment labelling and profiling

The technical cluster labels are converted into recruiter-facing business segment names for interpretation. These names are based on the customer behaviours observed in each cluster profile.

In [ ]:
segments = feature_df[["customer_number"]].copy()
segments["cluster"] = cluster_labels

# Business labels matched to the stable random_state=42 K-Means solution used in the portfolio report.
business_labels = {
    0: "Mainstream Value Shoppers",
    1: "Premium Mission Shoppers",
    2: "Convenience Top-Up Regulars",
    3: "Habitual Micro-Trip Shoppers",
    4: "Periodic Stock-Up Shoppers",
}

segments["segment_name"] = segments["cluster"].map(business_labels)

segment_sizes = (
    segments["segment_name"]
    .value_counts()
    .rename_axis("segment_name")
    .reset_index(name="n_customers")
)
segment_sizes["pct_customers"] = (segment_sizes["n_customers"] / segment_sizes["n_customers"].sum() * 100).round(1)
segment_sizes

In [ ]:
profile_df = feature_df.copy()
profile_df["segment_name"] = segments["segment_name"].values

key_kpis = [
    "n_baskets", "recency_days",
    "total_basket_spend", "avg_basket_spend", "max_basket_spend",
    "total_basket_qty", "avg_basket_qty", "avg_basket_categories",
    "fruit_veg", "dairy", "grocery_food", "confectionary",
    "drinks", "tobacco", "lottery", "cashpoint", "bakery",
]
key_kpis = [col for col in key_kpis if col in profile_df.columns]

profile_mean = profile_df.groupby("segment_name")[key_kpis].mean().round(2)
profile_median = profile_df.groupby("segment_name")[key_kpis].median().round(2)

profile_mean

In [ ]:
# Save profiling outputs
segment_sizes.to_csv(OUTPUT_DIR / "segment_sizes.csv", index=False)
profile_mean.to_csv(OUTPUT_DIR / "segment_profile_mean.csv")
profile_median.to_csv(OUTPUT_DIR / "segment_profile_median.csv")

segment_assignments = segments.rename(columns={"customer_number": "customer_id"})
segment_assignments[["customer_id", "segment_name"]].to_csv(
    OUTPUT_DIR / "customer_segment_assignments.csv", index=False
)

print("Saved profiling outputs to:", OUTPUT_DIR)

In [ ]:
# Segment size chart
plot_sizes = segment_sizes.sort_values("n_customers", ascending=True)

plt.figure(figsize=(8, 4.5))
plt.barh(plot_sizes["segment_name"], plot_sizes["n_customers"])
plt.xlabel("Number of customers")
plt.title("Segment sizes")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "segment_sizes.png", dpi=150)
plt.show()

In [ ]:
# Value gradient chart
value_gradient = profile_mean["total_basket_spend"].sort_values()

plt.figure(figsize=(8, 4.5))
plt.barh(value_gradient.index, value_gradient.values)
plt.xlabel("Mean total spend")
plt.title("Mean customer value by segment")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "value_gradient.png", dpi=150)
plt.show()

In [ ]:
# Standardised heatmap of segment profiles
z = profile_df[key_kpis].copy()
z = (z - z.mean()) / z.std(ddof=0)
z["segment_name"] = profile_df["segment_name"].values
z_means = z.groupby("segment_name")[key_kpis].mean()

plt.figure(figsize=(12, 4.5))
plt.imshow(z_means, aspect="auto")
plt.yticks(range(len(z_means.index)), z_means.index)
plt.xticks(range(len(key_kpis)), key_kpis, rotation=90)
plt.colorbar(label="Standardised mean")
plt.title("Segment profile heatmap")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "segment_profile_heatmap.png", dpi=150)
plt.show()

In [ ]:
# PCA scatter plot for visual separation
plt.figure(figsize=(7, 5))
plt.scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=cluster_labels, s=10, alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Customers in PCA space, coloured by cluster")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pca_cluster_scatter.png", dpi=150)
plt.show()

## 12. Business interpretation summary

The segmentation identifies five customer groups with different commercial roles:

1. **Mainstream Value Shoppers** — large low-to-mid value base; best handled with low-cost scalable campaigns.
2. **Premium Mission Shoppers** — small but highest-value group; priority for retention and service quality.
3. **Convenience Top-Up Regulars** — very frequent convenience shoppers with strong ancillary/category signals.
4. **Habitual Micro-Trip Shoppers** — high footfall but low basket value; opportunity is basket-building.
5. **Periodic Stock-Up Shoppers** — valuable larger-basket shoppers with longer gaps between trips; suitable for timed reactivation and stock-up offers.

The recommended target groups are **Premium Mission Shoppers** for retention and **Periodic Stock-Up Shoppers** for reactivation and basket-building.

## 13. Limitations and next steps

- The segmentation is descriptive and exploratory; it does not prove causal impact of marketing actions.
- Raw transaction data covers a six-month sample, so seasonal effects may be incomplete.
- K-Means assumes approximately spherical clusters in the transformed feature space; alternative methods could be compared in future work.
- The next professional step would be campaign testing or uplift modelling to verify whether each recommended intervention improves incremental revenue.